In [ ]:
!wget "https://fem-on-colab.github.io/releases/fenics-install-real.sh" -O "/tmp/fenics-install.sh" && bash "/tmp/fenics-install.sh"
!pip install meshio

# Soluzione di riferimento con FEM

Sia il dominio associato all'artefatto in analisi.

In questa fase, andremo a considerare sul dominio il modello associato al monitoraggio della temperatura sull'artefatto, con condizioni iniziali e condizioni al contorno fissate tramite i dati acquisiti con *sensori simulati*.

Effettuiamo gli import necessari

In [3]:
import json
import meshio
import numpy as np
import pandas as pd
import fenics as fe
import matplotlib.pyplot as plt

## Acquisizione del dominio

In [4]:
# Path to the XDMF file
file_path = "1_Base_Roccia.xdmf"

# Create an XDMFFile object for reading
xdmf_file = fe.XDMFFile(file_path)

# Create an empty Mesh object to store the mesh data
mesh = fe.Mesh()

# Read the mesh data from the XDMF file
xdmf_file.read(mesh)

# Close the XDMFFile object after reading
xdmf_file.close()

# Carica la mesh da file XDMF
mesh = fe.Mesh()
with fe.XDMFFile("1_Base_Roccia.xdmf") as infile:
    infile.read(mesh)

## Definizione dello spazio funzionale

In [5]:
V = fe.FunctionSpace(mesh, 'CG', 1)

## Definizione delle condizioni al contorno

In [6]:
# Acquisizione dati iniziali e al contorno
df = pd.read_csv("0_Base_TempData.csv", sep=";")

bc_elements = df["Temperature_C"].values

## Definizione numero di step

In [8]:
# Numero di passi temporali e intervallo di tempo
num_steps = bc_elements.shape[0]
T = 1.0
dt = T / (num_steps-1)

## Definizione delle variabili di stato e termine omogeneo

In [10]:
# Relativo alle condizioni iniziali
u_n = fe.interpolate(fe.Constant(bc_elements[0]), V)

u = fe.TrialFunction(V)
v = fe.TestFunction(V)
# Definizione della forma bilineare e lineare per l'equazione del calore
k = fe.Constant(1.0)  # Conduttività termica
a = u*v*fe.dx + dt*k*fe.dot(fe.grad(u), fe.grad(v))*fe.dx

#L = (u_n + fe.Constant(0.0)*dt) * v * fe.dx  # Senza sorgente di calore
L = u_n * v * fe.dx

## Risoluzione step-by-step

In [13]:
xdmf_sol = fe.XDMFFile("2_Base_FEM_Solution.xdmf")
xdmf_sol.parameters["flush_output"] = True
xdmf_sol.parameters["functions_share_mesh"] = True

u = fe.interpolate(fe.Constant(bc_elements[0]), V)

xdmf_sol.write(u, 0)

memorize_solution = dict()

# Estrazione delle coordinate dei punti della mesh
mesh_coordinates = mesh.coordinates()

# Valutazione della soluzione nei punti della mesh
solution_values = u_n.compute_vertex_values(mesh)

memorize_solution[0] = {
    "temporal_point" : 0,
    "spatial_points" : mesh_coordinates,
    "solution" : solution_values
}

u = fe.Function(V)

# Risoluzione del problema ad ogni passo temporale
for step in range(1, num_steps):
    # Istante temporale
    t = dt*step

    # Definizione del valore costante al contorno per l'iterazione corrente
    contour_value = bc_elements[step]

    # Definizione delle condizioni al contorno
    bc = fe.DirichletBC(V, fe.Constant(contour_value), 'on_boundary')

    # Risoluzione del problema al passo temporale corrente
    fe.solve(a == L, u, bc)

    # Update previous solution
    u_n.assign(u)

    # Memorizzo
    xdmf_sol.write(u, t)

    # Estrazione delle coordinate dei punti della mesh
    mesh_coordinates = mesh.coordinates()

    # Valutazione della soluzione nei punti della mesh
    solution_values = u.compute_vertex_values(mesh)

    memorize_solution[step] = {
        "temporal_point" : t,
        "spatial_points" : mesh_coordinates,
        "solution" : solution_values
    }

In [14]:
memorize = np.zeros((int(num_steps*mesh_coordinates.shape[0]), int(1 + mesh_coordinates.shape[1] + 1)))
step = 1
for key, value in memorize_solution.items():
    memorize[(step-1)*mesh_coordinates.shape[0]: step*mesh_coordinates.shape[0], :] = np.concatenate(
        [
          value["spatial_points"],
          value["temporal_point"]*np.ones((mesh_coordinates.shape[0], 1)),
          value["solution"].reshape(value["solution"].shape[0], 1)
        ],
        axis=-1
    )
    step += 1

In [15]:
info = {
    "steps" : num_steps,
    "single_grid_dimension" : mesh_coordinates.shape[0]
}

with open("2_Base_info_solution_fem.json", "w") as fp:
    json.dump(info, fp)

# Selezione dei dati di training per PINN da mesh

Inseriamo dati temporali, spaziali e soluzione all'interno di un dataframe

In [16]:
# Salvataggio dati
df = pd.DataFrame(memorize, columns=["x1", "x2", "x3", "t", "u"])

In [17]:
num_train = 10_000
num_test = 10_000

sampled_df = df.sample(num_train+num_test, replace=False)

train_df = sampled_df.iloc[:num_train]
test_df = sampled_df.iloc[num_train:]

Salvataggio di tutto il dataset, dei punti di training, e dei punti di testing

In [18]:
df.to_csv("2_Base_FEM_Solution.csv", sep=";", index=False)
train_df.to_csv("2_Base_FEM_Train.csv", sep=";", index=False)
test_df.to_csv("2_Base_FEM_Test.csv", sep=";", index=False)

## Salvataggio soluzione

In [19]:
# Save solution
output_file = fe.HDF5File(mesh.mpi_comm(), "2_Base_FEM_u.h5", "w")
output_file.write(u, "2_Base_FEM_u")
output_file.close()

Esempio di load

In [ ]:
# Load solution
U_loaded = fe.Function(V)
input_file = fe.HDF5File(mesh.mpi_comm(), "2_Base_FEM_u.h5", "r")
input_file.read(U, "2_Base_FEM_u")
input_file.close()